# Semi-Parametric Bootstrap Cross-Fitting

This workflow implements a robust method for estimating the individualized probability of an outcome (AUC) using the paper's **two-stage semiparametric estimator** combined with **bootstrap cross-fitting**.

The goal is to estimate $P(T_{control} > T_{case} | x)$—the probability that a Control patient has a better outcome (e.g., survival time, biological age) than a Case patient—without assuming that prediction errors follow a perfect Gaussian distribution.

### 1. Generative Model: Two-Stage Mean/Variance Estimator

Following the Supplementary Material's Algorithm 1 (the same procedure used throughout this repository for the point estimates in Table 1 and Figures 2-18 -- see `src/simulation/mlp_reg_data_simulation_multi.py`), the conditional distribution of $Y$ given $X$ is modeled as location-scale, $Y \sim \mathcal{N}(\mu(x), \sigma^2(x))$, but $\mu$ and $\sigma$ are estimated by **two separately-trained networks**, not a single network with a joint likelihood:

1. **Stage 1 (mean):** a network $\hat\mu(x;\theta)$ is fit by minimizing squared error, $\theta = \arg\min_\theta \sum_i (y_i - \hat\mu(x_i;\theta))^2$.
2. **Stage 2 (variance):** the squared residuals of Stage 1, $\hat r_i^2 = (y_i - \hat\mu(x_i))^2$, become the *target* of a second network $\hat\sigma^2(x;\phi)$, again fit by minimizing squared error, $\phi = \arg\min_\phi \sum_i (\hat r_i^2 - \hat\sigma^2(x_i;\phi))^2$.

Using two separate MSE-trained networks instead of one jointly-trained heteroscedastic network (the classic Nix & Weigend construction) keeps this uncertainty-quantification step consistent with every other result already reported in the paper, which uses Algorithm 1's two-stage procedure throughout -- rather than introducing a third, different estimation procedure just for the confidence bands.

---

### 2. Bootstrap Cross-Fitting (Residual Collection)

To perform honest simulations, we need the **true distribution of model errors** ($\epsilon$), not the theoretical one. We cannot use training errors because they are biased (underestimated due to overfitting) -- indeed, the point-estimate procedure above computes Stage 2's residuals on the *same* data Stage 1 was trained on, which has exactly this in-sample bias. We use a nested procedure to collect "clean" residuals instead:

**Algorithm:**

1.  **Outer Loop (Bootstrap $b=1 \dots B$):**
    Generate a resampled dataset $D^*_b$ from the original data $D$ (sampling with replacement).

2.  **Inner Loop (Cross-Fitting with $K$-Folds):**
    Split $D^*_b$ into $K$ folds. For each fold $k$:
    * **Train:** Fit the Stage 1 + Stage 2 model pair $M_k$ on the training partition $D_{train}^{(k)}$ (Stage 2 itself trained on Stage 1's in-fold-train residuals only).
    * **Validate (Cross-Fit):** Use $M_k$ to predict on the validation partition $D_{val}^{(k)}$ (data not seen by $M_k$).

    Calculate the **Standardized Empirical Residuals** ($\hat{\epsilon}$) only on the validation data:

    $$
    \hat{\epsilon}_j = \frac{y_j - \hat{\mu}(x_j)}{\hat{\sigma}(x_j)}, \quad \forall j \in D_{val}^{(k)}
    $$

    By the end of the $K$-folds, we obtain a pool of residuals $\mathcal{E}_b = \{ \hat{\epsilon}_1, \dots, \hat{\epsilon}_N \}$ representing the real error distribution on unseen data.

---

### 3. Semi-Parametric Density Estimation (Monte Carlo)

To predict the outcome for a new patient, we do not assume the error is Gaussian. Instead, we use the empirical distribution $\mathcal{E}$ collected in the previous step.

Let $x_0$ be a Control patient and $x_1$ be a Case patient.

**Monte Carlo Simulation:**
We generate $M$ possible scenarios (e.g., $M=500$) by sampling random residuals $\epsilon^*$ from the pool $\mathcal{E}$:

$$
T_0^{(m)} = \hat{\mu}(x_0) + \hat{\sigma}(x_0) \cdot \epsilon^*_A, \quad \epsilon^*_A \sim \text{Uniform}(\mathcal{E})
$$

$$
T_1^{(m)} = \hat{\mu}(x_1) + \hat{\sigma}(x_1) \cdot \epsilon^*_B, \quad \epsilon^*_B \sim \text{Uniform}(\mathcal{E})
$$

**AUC Calculation:**
The individualized AUC is estimated as the frequency with which the Control outcome exceeds the Case outcome across simulations:

$$
\widehat{AUC}(x_0, x_1) = \frac{1}{M} \sum_{m=1}^{M} \mathbb{I}\left( T_0^{(m)} > T_1^{(m)} \right)
$$

*Where $\mathbb{I}(\cdot)$ is the indicator function.*

---

### 4. Out-Of-Bag (OOB) Evaluation

To report a globally unbiased metric, we ensure that no patient is evaluated using a model that saw them during training. We leverage the properties of Bootstrap aggregation.

Let $I_b$ be the set of patient indices included in the training of Bootstrap $b$. For a specific patient $i$:

1.  Identify the Bootstrap iterations where patient $i$ was **excluded** (Out-Of-Bag, $i \notin I_b$).
2.  Average the AUC estimates obtained only from those iterations.

$$
\text{AUC\_Final}_i = \mathbb{E}_{b} \left[ \widehat{AUC}_b(x_i) \mid i \notin I_b \right]
$$

This approach captures both **aleatoric uncertainty** (via $\sigma$) and **epistemic uncertainty** (via Bootstrap), providing a robust and conservative estimate of the model's discriminative power.

The bootstrap/cross-fitting/OOB/Monte-Carlo-AUC machinery itself (steps 2-4) lives in `src/simulation/bootstrap_crossfit_oob.py`, shared with the simulation-study coverage analysis in `src/simulation/coverage_bootstrap_crossfit.py` -- this notebook only handles the NHANES-specific data loading and plotting.

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import os
import matplotlib.pyplot as plt

# num_to_onehot/normalize_column are identical to the ones already in
# src/real_data/data_reg_real.py (used by the real-data mlp_reg.py pipeline) --
# reused from there instead of duplicating them here. Run this notebook from the
# repository root so this path resolves.
sys.path.insert(0, os.path.join("src", "real_data"))
from data_reg_real import num_to_onehot, normalize_column

# bootstrap_crossfit_oob_shared implements the bootstrap + K-fold cross-fitting +
# OOB engine (Sections 2-4 of the markdown above) on top of the same two-stage
# mean/variance estimator (Section 1) used for every other FNN result in the
# paper, instead of a separately-trained heteroscedastic network.
sys.path.insert(0, os.path.join("src", "simulation"))
from bootstrap_crossfit_oob import bootstrap_crossfit_oob_shared

# =============================================================================
# 0. GLOBAL CONFIGURATION & REPRODUCIBILITY
# =============================================================================

# Set seeds to ensure results are the same every time this script runs
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Configuration dictionary acting as a replacement for command-line arguments.
# This controls the entire pipeline (input/output, model hyperparameters, etc.)
config = {
    "input_dir": "./data",           # Directory containing df_f.csv / df_m.csv
    "output_file": "./output",       # Root directory for saving CSV results and PNG plots (git-ignored)
    "combination": 6,                # Index selector for 'get_feature_combination'
    "k_folds": 2,                    # Number of splits for Cross-Validation
    "num_epochs": 500,               # Training iterations per fold
    "batch_size": 200,               # Number of samples per gradient update
    "learning_rate": 0.001,          # Step size for the optimizer
    "weight_decay": 1e-4,            # L2 Regularization (prevents overfitting)
    "dropout": 0.2,                  # Fraction of neurons to drop during training
    "hidden_layers": [64, 32],       # Architecture: Input -> 64 -> 32 -> Output
    "early_stop_patience": 20,       # Epochs of no val-loss improvement before stopping each fold's training
}

# Constants for NHANES data processing
GENDER_COL = "RIAGENDR"
GENDER_MAP = {"male": 1, "female": 2}
TARGET_COL = "TAC2"  # The biological age or time-to-event variable
GROUP_COLUMNS = ["tres", "cinco", "ocho"]  # Classification thresholds (e.g., survival years)

# Explicit filename per gender (data/df_f.csv, data/df_m.csv). This used to be a
# fuzzy filename-substring filter over every file in --input_dir, but --input_dir
# is now the shared data/ folder (also holding female_data.csv,
# female_data_with_cancer.csv, etc.), whose names also match a "female"/"male"
# substring filter -- the fuzzy filter would silently pick up the wrong files.
GENDER_FILES = {"female": "df_f.csv", "male": "df_m.csv"}

# =============================================================================
# 1. FEATURE SELECTION & FILTERS
# =============================================================================

def get_feature_combination(idx: int) -> list[str]:
    """
    Selects a specific list of input features (column names) based on an index.
    Useful for experimenting with different subsets of biological/demographic data.

    Args:
        idx (int): The identifier for the feature set (0-7).

    Returns:
        list[str]: A list of column names to be used as X (inputs).
    """
    if idx == 0:
        return ['RIDAGEYR.x', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1',
                'BPXSY1', 'BPXPLS', 'LBDSCHSI_43', 'LBXSTR_43', 'LBXSGL_43', 'RIAGENDR', 'LBXGH_39']
    elif idx == 1:
        return ['RIDAGEYR.x', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1',
                'BPXSY1', 'BPXPLS', 'BPXDI1', 'LBDSCHSI_43', 'LBXSTR_43', 'LBXSGL_43', 'RIAGENDR']
    # ... (skipping indices 2, 3, 4, 5 for brevity, logic remains the same)
    elif idx == 6:
        # Minimalist set: Age and BMI only
        return ['RIDAGEYR.x', 'BMI']
    elif idx == 7:
        # Physical measurements only
        return ['BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'BPXDI1']
    else:
        # Default fallback (same as idx 0)
        return ['RIDAGEYR.x', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1',
                'BPXSY1', 'BPXPLS', 'LBDSCHSI_43', 'LBXSTR_43', 'LBXSGL_43', 'RIAGENDR', 'LBXGH_39']

# =============================================================================
# 2. DATA PREPROCESSING HELPERS
# =============================================================================

def create_dict_from_df(df, index_0, index_1, output_var, variables):
    """
    Core data structuring function.
    1. Normalizes continuous variables.
    2. One-hot encodes categorical variables (if any).
    3. Stacks them into a single input matrix X.
    4. Splits data into dictionaries for 'All', 'Group 0' (Control), and 'Group 1' (Case).

    Unlike src/real_data/data_reg_real.py's create_dict, this does not require a
    'SEQN' subject-id column or NHANES survey weights ('wtmec4yr_adj_norm') -- df_f.csv/
    df_m.csv have neither, so weights default to 1 per subject when no 'weights'
    column is present. Kept notebook-local rather than merged into data_reg_real.py
    for that reason.

    Args:
        df (pd.DataFrame): The source dataframe.
        index_0 (array): Row indices belonging to Group 0 (e.g., Controls).
        index_1 (array): Row indices belonging to Group 1 (e.g., Cases).
        output_var (str): The name of the target column (Y).
        variables (list): List of feature column names (X).

    Returns:
        tuple: (dict_all, dict_0, dict_1). Each dict contains 'x', 'y', 'w' arrays.
    """
    L, colnames = [], []
    
    # Currently assuming all variables passed are continuous (interval)
    # If categorical variables were passed, they should be moved to 'cat_variables'
    int_variables = variables
    cat_variables = [] 
    
    for i, c in enumerate(variables):
        if c in int_variables:
            feature_vector = normalize_column(df[c])
            L.extend([np.expand_dims(feature_vector, 1)])
            colnames.extend([c])
        elif c in cat_variables:
            feature_vector = num_to_onehot(df[c].to_numpy())
            L.extend([feature_vector])
            for j in range(feature_vector.shape[1]):
                colnames.extend([c + "_" + str(j)])

    # Horizontal stack: Combine all feature columns into one matrix
    input_data = np.hstack(L).astype('float32')
    
    # Prepare Target (Y) and Weights (W)
    target = df[output_var].values.astype('float32').reshape(-1, 1)
    weights = df['weights'].astype('float32').values if 'weights' in df.columns else np.ones(len(df), dtype='float32')
    weights = weights.reshape(-1, 1)

    # Create the dictionaries
    dict_all = {'x': input_data, 'y': target, 'w': weights}
    dict_0   = {'x': input_data[index_0], 'y': target[index_0], 'w': weights[index_0]}
    dict_1   = {'x': input_data[index_1], 'y': target[index_1], 'w': weights[index_1]}
    
    return dict_all, dict_0, dict_1

def load_data_safe(df, variables, output_var, group_col):
    """
    Wrapper that identifies the indices for Group 0 and Group 1 based on the 'group_col'.
    """
    index_0 = np.where((df[group_col] == 0))[0]
    index_1 = np.where((df[group_col] == 1))[0]
    return create_dict_from_df(df, index_0, index_1, output_var, variables)

# =============================================================================
# 3. MAIN WORKFLOW: BOOTSTRAP + CROSS-FITTING
# =============================================================================

def run_bootstrap_crossfitting(group_col, gender, config, B=100, K=5):
    """
    Orchestrates the entire evaluation process.

    Process:
    1. Loads files.
    2. Delegates the bootstrap + K-fold cross-fitting + OOB aggregation to
       bootstrap_crossfit_oob_shared (two-stage mean/variance estimator, same
       procedure used for every other FNN result in the paper).
    3. Saves/plots the resulting per-row AUC mean/CI.

    Args:
        group_col (str): The column used to split groups (e.g., 'three_year_survival').
        gender (str): Gender filter.
        B (int): Number of bootstraps.
        K (int): Number of CV folds.
    """
    print(f"\n=== Bootstrap-CrossFitting (Two-Stage Estimator): {group_col} - {gender} (B={B}, K={K}) ===")
    
    combination_idx = config.get("combination", 5)
    combination = get_feature_combination(combination_idx)
    target = 'TAC2'
    
    # File handling: read the one file for this gender directly (see GENDER_FILES).
    if gender not in GENDER_FILES:
        raise ValueError(f"Unknown gender {gender!r}; expected one of {list(GENDER_FILES)}")
    onlyfiles = [GENDER_FILES[gender]]
        
    # Device setup (Mac Metal vs CUDA vs CPU)
    if torch.cuda.is_available(): device = torch.device("cuda")
    elif torch.backends.mps.is_available(): device = torch.device("mps")
    else: device = torch.device("cpu")
    print(f" > Device: {device}")
    
    for f in onlyfiles:
        file_path = os.path.join(config["input_dir"], f)
        
        # Load and clean Data
        df_raw = pd.read_csv(file_path)
        df = df_raw.dropna(subset=combination + [target]).reset_index(drop=True)
        if len(df) < 10: continue

        # Prepare full dataset tensors
        data_all, data_0, data_1 = load_data_safe(df, combination, target, group_col)
        
        X_eval = data_all['x']
        Y_all = df[group_col].values
        N_eval = len(df)
        
        print(f" > Dataset: {os.path.basename(file_path)} | N={N_eval}")
        
        auc_mean, auc_lower, auc_upper, n_oob = bootstrap_crossfit_oob_shared(
            data_all['x'], data_all['y'], Y_all, data_all['w'],
            X_eval, config, B, K, device, n_mc=500,
        )
        
        # Save to CSV
        df_res = df.copy()
        df_res['CROC_AUC_Mean'] = auc_mean
        df_res['CROC_CI_Lower'] = auc_lower
        df_res['CROC_CI_Upper'] = auc_upper
        df_res['CROC_N_OOB'] = n_oob
        
        out_dir = os.path.join(config["output_file"], group_col, os.path.splitext(f)[0], f"BootstrapOOB_Empirical_B{B}")
        os.makedirs(out_dir, exist_ok=True)
        df_res.to_csv(os.path.join(out_dir, "results_{}_{}.csv".format(group_col, gender)), index=False)
        
        # --- PLOTTING (Gaussian Smoothing) ---
        group_map = {
            "tres": "three",
            "cinco": "five",
            "ocho": "eight"
        }
        col_edad = 'RIDAGEYR.x' if 'RIDAGEYR.x' in df_res.columns else 'RIDAGEYR'
        if col_edad in df_res.columns and config.get("show_plots", True):
            
            # Group by Age to get raw means
            df_grouped = df_res.groupby(col_edad)[['CROC_AUC_Mean', 'CROC_CI_Lower', 'CROC_CI_Upper']].mean()
            
            # Apply Gaussian Rolling Window for professional smoothing
            window_size = 15
            sigma_val = 3.0
            df_smooth = df_grouped.rolling(window=window_size, win_type='gaussian', center=True, min_periods=3).mean(std=sigma_val)
            
            plt.figure(figsize=(10, 7))
            plt.fill_between(df_smooth.index, 
                             df_smooth['CROC_CI_Lower'], 
                             df_smooth['CROC_CI_Upper'], 
                             color='blue', alpha=0.15, label='95% CI (Smoothed)')
            plt.plot(df_smooth.index, df_smooth['CROC_AUC_Mean'], 
                     color='blue', linewidth=2.5, label='AUC Trend')
            
            plt.xlabel("Age (Years)")
            plt.ylabel("Estimated AUC")
            plt.title(f"Bootstrap OOB: {gender} {group_map.get(group_col, group_col)} survival years")
            plt.legend()
            plt.savefig(os.path.join(out_dir, "auc_plot_smooth_{}_{}.png".format(group_col, gender)), dpi=300)
            plt.show()
            plt.close()

In [ ]:
run_bootstrap_crossfitting('tres', 'female', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('tres', 'male', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('cinco', 'female', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('cinco', 'male', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('ocho', 'female', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('ocho', 'male', config, B=100, K=5)